In [31]:
import os
import pandas as pd
from importlib import reload
import numpy as np
import seaborn as sns
from tqdm import *
import scipy.special
from numpy.linalg import eigh
from joblib import Parallel, delayed
import matplotlib.pyplot as plt

In [32]:
def concat_model_4fold(test_path, res_path):
    market = 'ALL'
    model_res = []
    for valid_period in ['validperiod1', 'validperiod2', 'validperiod3', 'validperiod4']:
        data_path = rf"{test_path}/{market}-{valid_period}"
        date_list = list(set([x[:8] for x in os.listdir(data_path)]))
        date_list.sort()
        all_res = []
        for date in date_list:
            date_res = pd.read_pickle(f'{data_path}/{date}.pkl').T
            date_res.index = [date] * len(date_res)
            all_res.append(date_res)
        all_res = pd.concat(all_res, axis=0).sort_index()
        all_res.index.name = "date"
        model_res.append(all_res.apply(lambda x: (x - x.mean()) / x.std(), axis=1))
    model_res = sum(model_res)
    model_res.reset_index().to_feather(rf"{res_path}/{market}_zscore_score.fea")
    return model_res

# 根据模型打分获取对应的label_ret和ic
def get_ret_ic(score, ret_data, liquid_data, start='20230101', end='20241231', money=1.5e9, print_quarterly=False):
    model_score = score.copy()
    label_ret = []
    ic = []
    datelist = []
    # 这是一段日度调仓计算日度收益率的代码
    for date in model_score.loc[start:end].index:
        datelist.append(date)
        code_rank = model_score.loc[date].sort_values(ascending=False)
        ret = ret_data.loc[date].reindex(code_rank.index).fillna(0) * 100
        liquid = liquid_data.loc[date].reindex(code_rank.index).fillna(0)
        total_hold = 0
        total_earned = 0
        for num,code in enumerate(code_rank.index):
            if num >= 500:
                break
            if (money - total_hold) < 1:
                break
            hold_money = min(money - total_hold, liquid[code])
            total_hold += hold_money
            total_earned += ret[code] * hold_money
        total_ret = total_earned / money
        label_ret.append(total_ret)
        ic.append(code_rank.corr(ret))
    ic = pd.Series(ic, index=datelist, dtype='float')
    label_ret = pd.Series(label_ret, index=datelist, dtype='float')
    if print_quarterly:
        print_quarterly_metrics(label_ret, ic)
    return label_ret, ic

def print_quarterly_metrics(ret, ic):
    """打印每个季度的收益和IC指标"""
    ret_series = ret.copy()
    ic_series = ic.copy()
    ret_series.index = pd.to_datetime(ret_series.index)
    ic_series.index = pd.to_datetime(ic_series.index)
    quarterly_ret = ret_series.groupby(pd.Grouper(freq='QE')).mean()
    quarterly_ic = ic_series.groupby(pd.Grouper(freq='QE')).mean()
    quarterly_ret.index = [f"{idx.year}Q{idx.quarter}" for idx in quarterly_ret.index]
    quarterly_ic.index = [f"{idx.year}Q{idx.quarter}" for idx in quarterly_ic.index]
    quarterly_df = pd.DataFrame({
        '季度平均收益(%)': quarterly_ret.round(4),
        '季度平均IC': quarterly_ic.round(4)
    })
    print("\n===== 季度表现指标 =====")
    print(quarterly_df)
    print("=======================\n")

def get_metrics(score, ret_data, liquid_data, start='20230101', end='20241231', money=1.5e9, print_quarterly=True):
    ret_list, ic_list = get_ret_ic(score, ret_data, liquid_data, start=start, end=end, money=money, print_quarterly=print_quarterly)
    # 当前模型评价
    score_metrics = {
        "IC": ic_list.mean(),
        "ICIR": ic_list.mean() / ic_list.std() if ic_list.std() != 0 else 0,
        "top_return": ret_list.mean(),
        "top_return_stability": ret_list.mean() / ret_list.std() if ret_list.std() != 0 else 0
    }
    return score_metrics

def print_metrics(model_metrics, bench_metrics=None, title="模型评估结果"):
    metric_keys = list(model_metrics.keys())
    df = pd.DataFrame(index=metric_keys)
    df.index.name = "指标"
    df["模型值"] = [model_metrics[k] for k in metric_keys]
    if bench_metrics is not None:
        df["基准值"] = [bench_metrics.get(k, None) for k in metric_keys]
        df["提升值"] = ((df["模型值"] - df["基准值"]) / df["基准值"]) * 100
    print(f"\n{title}")
    print(df.round(4))

def plot_model(model_score, bench_score, ret_data, liquid_data, start='20230101', end='20241231', money=1.5e9):
    model_ret, _ = get_ret_ic(model_score, ret_data, liquid_data, start=start, end=end, money=money, print_quarterly=False)
    model_ret = model_ret/100
    bench_ret, _ = get_ret_ic(bench_score, ret_data, liquid_data, start=start, end=end, money=money, print_quarterly=False)
    bench_ret = bench_ret/100

    dates = model_ret.index.tolist()
    seen_months = set()
    month_ticks = []
    month_labels = []
    for i, d in enumerate(dates):
        month = d[:6] 
        if month not in seen_months:
            seen_months.add(month)
            month_ticks.append(i)
            month_labels.append(d)

    # 1. 绘制每日收益率曲线
    plt.figure(figsize=(12, 6))
    plt.plot(dates, model_ret.cumsum(), label='model', color='blue')
    plt.plot(dates, bench_ret.cumsum(), label='bench', color='orange')
    plt.title('Cumulative Return Of Model & Bench', fontsize=14)
    plt.ylabel('Cumulative Return(%)', fontsize=12)
    plt.xticks(ticks=month_ticks, labels=month_labels, rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    # 2. 绘制相对提升
    plt.figure(figsize=(12, 6))
    relative_improvement = model_ret - bench_ret
    plt.plot(dates, relative_improvement.cumsum(), label='model', color='blue')
    plt.title('Relative Iprovement', fontsize=14)
    plt.ylabel('Relative Iprovement(%)', fontsize=12)
    plt.xticks(ticks=month_ticks, labels=month_labels, rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()

def diversity_entropy(score):
    """
    计算 DE
    """
    if isinstance(score, pd.DataFrame):
        score = score.values
    m = score.shape[1]
    cov_matrix = np.cov(score, rowvar=False)
    eigenvalues, _ = eigh(cov_matrix)
    eigenvalues = np.sort(eigenvalues)[::-1]
    normalized_eigenvalues = eigenvalues / (np.sum(eigenvalues) + 1e-10)
    entropy = 0
    for p in normalized_eigenvalues:
        if p > 1e-10:  # 只对非零概率计算
            entropy -= p * np.log(p)
    de = entropy / np.log(m)
    return de

def ensemble_scores(*dfs):
    """
    输入：任意数量的打分 DataFrame（index=日期，columns=股票）
    输出：拼接后的 DataFrame，columns=score1, score2, ...
    """
    result_list = []
    for i, df in enumerate(dfs, 1):
        df_norm = df.copy()
        df_norm = df_norm.apply(lambda x: (x - x.mean()) / x.std(), axis=1)
        df_norm = df_norm.stack().rename(f"score{i}")
        result_list.append(df_norm)
    result = pd.concat(result_list, axis=1)
    return result.dropna()


In [33]:
# 标签
label_path = rf'/home/user96/label_data1'
label_name = rf'label1'
# 流动性数据
liquid_path = rf'/home/user96/label_data1'
liquid_name = rf'can_trade_amt1'
class params:
    liquid_data = pd.read_feather(rf"{liquid_path}/{liquid_name}.fea").set_index("index")
    ret_data = pd.read_feather(rf"{label_path}/{label_name}.fea").set_index("index")
    
# bench打分评估
bench1 = pd.read_feather(rf'/home/user96/score_file/bench1_20250822.fea').set_index('date')
bench2 = pd.read_feather(rf'/home/user96/score_file/bench2_20250822.fea').set_index('date')
bench3 = pd.read_feather(rf'/home/user96/score_file/bench3_20250822.fea').set_index('date')
bench4 = pd.read_feather(rf'/home/user96/score_file/bench4_20250822.fea').set_index('date')

# 模型路径与结果路径
root_path = rf'/home/user96/20251010_MLP2'
model_test_path = rf'/home/user96/20251010_MLP2/model_test/nn--fac20250212_fill_zero--label1--dropout0.2'
model_res_path = rf'{root_path}/model_res'
for path in [model_res_path]:
    os.makedirs(path, exist_ok=True)

# 收集模型训练结果
score1 = concat_model_4fold(test_path=model_test_path, res_path=model_res_path) 

bench_new = ensemble_scores(bench1, bench2, bench3, bench4, score1)
bench_new = bench_new.mean(axis=1).unstack()
score1.columns.name = None

In [34]:

def get_topk(x, k=10, axis=1):
    """
    返回每行/每列的 top-k 的索引列表（axis=1: 每行取列标签；axis=0: 每列取行标签）。
    兼容 DataFrame / Series。会把数值统一上采样到 float64，避免 float16 造成的 nlargest 异常。
    """
    # 传入是 Series
    if isinstance(x, pd.Series):
        s = pd.to_numeric(x, errors='coerce').astype('float64')
        s = s.replace([np.inf, -np.inf], np.nan).dropna()
        if s.empty:
            return []
        return s.nlargest(min(k, s.size)).index.tolist()

    # 传入是 DataFrame
    if isinstance(x, pd.DataFrame):
        df_num = x.select_dtypes(include=[np.number]).copy()
        # 统一到 float64，避免 float16 触发底层算法类型问题
        df_num = df_num.apply(pd.to_numeric, errors='coerce').astype('float64')
        df_num = df_num.replace([np.inf, -np.inf], np.nan)

        if axis == 1:  # 每行取列的 top-k
            return df_num.apply(
                lambda row: row.dropna().nlargest(min(k, int(row.count()))).index.tolist(),
                axis=1
            )
        elif axis == 0:  # 每列取行的 top-k（如需）
            return df_num.T.apply(
                lambda col: col.dropna().nlargest(min(k, int(col.count()))).index.tolist(),
                axis=1
            )
        else:
            raise ValueError("axis 只能是 0 或 1")

    raise TypeError("get_topk 仅支持 pandas DataFrame 或 Series")


k = 500
topk_ret = get_topk(params.ret_data, k)
topk_bench1 = get_topk(bench1, k)
topk_bench2 = get_topk(bench2, k)
topk_bench3 = get_topk(bench3, k)
topk_bench4 = get_topk(bench4, k)
topk_model = get_topk(score1, k)
topk_new = get_topk(bench_new, k)

In [35]:
import pandas as pd

def calc_hit_rate(A: pd.Series, B: pd.Series, return_daily=False):
    """
    计算B对A的命中率。
    
    参数：
        A, B: pandas.Series，索引为日期（int或str），值为list或类似结构。
        return_daily: 是否返回每日命中率（默认只返回整体平均命中率）
    
    返回：
        若 return_daily=True -> pandas.Series（每日命中率）
        否则 -> float（整体平均命中率）
    """
    # 对齐日期（取共同的日期区间）
    common_dates = A.index.intersection(B.index)
    A = A.loc[common_dates]
    B = B.loc[common_dates]

    # 计算每日命中率
    daily_hit_rate = []
    for date in common_dates:
        set_a = set(A.loc[date])
        set_b = set(B.loc[date])
        if len(set_b) == 0:
            daily_hit_rate.append(0)
        else:
            hit = len(set_a & set_b) / len(set_b)
            daily_hit_rate.append(hit)

    daily_hit_rate = pd.Series(daily_hit_rate, index=common_dates)

    if return_daily:
        return daily_hit_rate
    else:
        return daily_hit_rate.mean()
# 计算整体命中率
hit_rate_bench1 = calc_hit_rate(topk_ret, topk_bench1)
hit_rate_bench2 = calc_hit_rate(topk_ret, topk_bench2)
hit_rate_bench3 = calc_hit_rate(topk_ret, topk_bench3)
hit_rate_bench4 = calc_hit_rate(topk_ret, topk_bench4)
hit_rate_model = calc_hit_rate(topk_ret, topk_model)
hit_rate_new = calc_hit_rate(topk_ret, topk_new)

print("bench1整体命中率：", hit_rate_bench1)
print("bench2整体命中率：", hit_rate_bench2)
print("bench3整体命中率：", hit_rate_bench3)
print("bench4整体命中率：", hit_rate_bench4)
print("model整体命中率：", hit_rate_model)
print("new整体命中率：", hit_rate_new)


bench1整体命中率： 0.22964462809917355
bench2整体命中率： 0.2541115702479339
bench3整体命中率： 0.25072314049586775
bench4整体命中率： 0.2540123966942149
model整体命中率： 0.1590330578512397
new整体命中率： 0.25410330578512397
